# Phase 1 — Business Understanding

## Business Problem Framing & Success Criteria

### Business Context - Derived from Client Interviews and other Sources of Enterprise Information

West Chester University is seeking to better understand the factors that influence university graduation rates. As a university, West Chester has both fixed and variable operating costs, and maintaining a consistent number of students through graduation helps provide greater stability when planning for tuition revenue and other university-related revenue. The university would like to compare its performance with similar regional universities in order to better understand where opportunities for improvement may exist. For this analysis, West Chester Universitys peers will primarily consist of smaller regional universities in Pennsylvania and universities located within approximately 100 miles of West Chester. Examples of potential peer institutions include Millersville University, Shippensburg University, and Slippery Rock University.


### Business Problem

West Chester University needs to identify the factors that are most relevant in explaining and predicting graduation rates. The university also wants to understand how it compares with its regional peer institutions on those factors.
Identifying these factors would help the university determine where resource investments could potentially have the greatest impact on graduation rates.

### North Star

Every student who starts with a degree, and no student is left behind because of who they are or where they started. 

### Key Decision

Based on the factors that are most strongly associated with graduation rates, which areas should West Chester University prioritize for potential investment or intervention, and how does WCU compare with its regional peer institutions in those areas?

### Analytical Objective

Identify and quantify the factors that are most strongly associated with university graduation rates, and compare West Chester University with similar regional peer institutions to determine where meaningful differences and potential opportunities for improvement exist.


### Key Analytical Questions

- What characteristics are most strongly associated with higher graduation rate? 
- Can graduation rates be predicted with sufficient accuracy to support resource allocation to then further increase graduation rates? 
- What university traits are most strongly associated with higher graduation rates? 

### Analytical Approach

A data-driven analytical approach is used to examine graduation rates among universities within 100 miles of WCUPA and explore patterns associated with differences in graduation outcomes.

The analysis is designed to:

- Compare WCUPA's graduation rate with nearby peer institutions
- Examine relationships between graduation rates and university characteristics
- Identify factors associated with higher or lower graduation rates
- Support informed resource-allocation and student-success decisions

### Success Criteria

**Business Success — Alignment to the North Star**

- Identify how WCUPA's graduation rate compares with regional peers
- Identify university characteristics associated with differences in graduation rates
- Provide insights that may help inform resource-allocation decisions and student-success initiatives

**Analytical Success — How Analytics Supports Business Success**

- Establish a reliable dataset of universities within 100 miles of WCUPA
- Evaluate graduation rates using descriptive statistics and visualizations
- Examine relationships between graduation rates and selected university characteristics
- Interpret findings clearly while recognizing that associations do not establish causation

## Phase 2 — Exploratory Data Analysis (EDA) of target and predictors (Understanding the Data)

### 2.1 Hypotheses to Investigate

Based on the business problem and available employee data, the following hypotheses will guide the exploratory analysis. These hypotheses identify potential drivers of attrition and provide a foundation for evaluating opportunities for feature engineering and interaction effects.

### 2.2 Load Libraries 

In [18]:
# Data Manipulation
import pandas as pd
import numpy as np

# Visualization
import seaborn as sns
import matplotlib.pyplot as plt

# Notebook Display
from IPython.display import display

# Notebook Settings
sns.set(style="whitegrid")

import warnings
warnings.filterwarnings("ignore")

### 2.3 Load Data

In [19]:
df = pd.read_excel("/Users/zuhasadaqat/Desktop/Capstone for BUA 400/Spring_2026_data2.xlsx")
df.head()

,Unnamed: 0,Private,Apps,Accept,Enroll,Top10perc,Top25perc,F.Undergrad,P.Undergrad,Outstate,Room.Board,Books,Personal,PhD,Terminal,S.F.Ratio,perc.alumni,Expend,Grad.Rate
0,Abilene Christian University,Yes,1660,1232,721,23,52,2885,537,7440,3300,450,2200,70,78,18.1,12,7041,60
1,Adelphi University,Yes,2186,1924,512,16,29,2683,1227,12280,6450,750,1500,29,30,12.2,16,10527,56
2,Adrian College,Yes,1428,1097,336,22,50,1036,99,11250,3750,400,1165,53,66,12.9,30,8735,54
3,Agnes Scott College,Yes,417,349,137,60,89,510,63,12960,5450,450,875,92,97,7.7,37,19016,59
4,Alaska Pacific University,Yes,193,146,55,16,44,249,869,7560,4120,800,1500,76,72,11.9,2,10922,15


In [22]:
# Load the NCES location dataset
locations = pd.read_csv("/Users/zuhasadaqat/Desktop/Postsecondary_School_Locations_Current_-3554188794373885176.csv")

# Preview the dataset
print("Dataset size:", locations.shape)
display(locations.head())

Dataset size: (6438, 24)


,OBJECTID,School identification number,Name of institution,Reported street address,Reported city,Reported state,Reported ZIP code,State FIPS,County FIPS,County name,...,Core Based Statistical Area name,Metropolitan or Micropolitan Statistical Area indicator,Combined Statistical Area,Combined Statistical Area name,Congressional District,State Legislative District - Lower,State Legislative District - Upper,School Year,x,y
0,1,100654,Alabama A & M University,4900 Meridian Street,Normal,AL,35762,01,01089,Madison County,...,"Huntsville, AL",1,290,"Huntsville-Decatur-Albertville, AL-TN",0105,01019,01008,2025-2026,-86.568497,34.783361
1,2,100663,University of Alabama at Birmingham,Administration Bldg Suite 1070,Birmingham,AL,35294-0110,01,01073,Jefferson County,...,"Birmingham, AL",1,142,"Birmingham-Cullman-Talladega, AL",0107,01055,01018,2025-2026,-86.799340,33.505691
2,3,100690,Amridge University,1200 Taylor Rd,Montgomery,AL,36117-3553,01,01101,Montgomery County,...,"Montgomery, AL",1,388,"Montgomery-Selma, AL",0102,01078,01025,2025-2026,-86.174005,32.362603
3,4,100706,University of Alabama in Huntsville,301 Sparkman Dr,Huntsville,AL,35899,01,01089,Madison County,...,"Huntsville, AL",1,290,"Huntsville-Decatur-Albertville, AL-TN",0105,01053,01002,2025-2026,-86.640444,34.724550
4,5,100724,Alabama State University,915 S Jackson Street,Montgomery,AL,36104-0271,01,01101,Montgomery County,...,"Montgomery, AL",1,388,"Montgomery-Selma, AL",0102,01077,01026,2025-2026,-86.295672,32.364311


### 2.4 Data Structures and Data Types
We begin by reviewing the structure of the dataset to understand the information available for analysis.

This includes:

- Dataset size
- Column names
- Data types
- Summary statistics
- Geographical filtering

In [24]:

# Find West Chester University of Pennsylvania
wcupa = locations[
    locations['Name of institution']
    == 'West Chester University of Pennsylvania'
]

display(
    wcupa[[
        'Name of institution',
        'Latitude of school location',
        'Longitude of school location'
    ]]
)


# WCUPA coordinates
wcupa_lat = 39.953398
wcupa_lon = -75.598120

# Convert latitude and longitude to radians
lat1 = np.radians(wcupa_lat)
lon1 = np.radians(wcupa_lon)

lat2 = np.radians(locations['Latitude of school location'])
lon2 = np.radians(locations['Longitude of school location'])

# Calculate differences
dlat = lat2 - lat1
dlon = lon2 - lon1

# Haversine formula
a = (
    np.sin(dlat / 2) ** 2
    + np.cos(lat1) * np.cos(lat2)
    * np.sin(dlon / 2) ** 2
)

c = 2 * np.arcsin(np.sqrt(a))

# Earth's average radius in miles
locations['Distance_Miles'] = 3958.8 * c

# Display the calculated distances
display(
    locations[[
        'Name of institution',
        'Reported city',
        'Reported state',
        'Distance_Miles'
    ]].head(10)
)



# Keep institutions within 100 miles of WCUPA
locations_100mi = locations[
    locations['Distance_Miles'] <= 100
].copy()

print("Institutions in original location dataset:", len(locations))
print("Institutions within 100 miles:", len(locations_100mi))

# Display nearby institutions, closest first
display(
    locations_100mi[[
        'Name of institution',
        'Reported city',
        'Reported state',
        'Distance_Miles'
    ]].sort_values('Distance_Miles')
)


,Name of institution,Latitude of school location,Longitude of school location
3097,West Chester University of Pennsylvania,39.953398,-75.59812


,Name of institution,Reported city,Reported state,Distance_Miles
0,Alabama A & M University,Normal,AL,699.583893
1,University of Alabama at Birmingham,Birmingham,AL,762.692102
2,Amridge University,Montgomery,AL,788.347630
3,University of Alabama in Huntsville,Huntsville,AL,705.241929
4,Alabama State University,Montgomery,AL,793.327876
5,University of Alabama System Office,Tuscaloosa,AL,808.472864
6,The University of Alabama,Tuscaloosa,AL,809.003857
7,Central Alabama Community College,Alexander City,AL,751.816596
8,Athens State University,Athens,AL,717.441195
9,Auburn University at Montgomery,Montgomery,AL,788.263631


Institutions in original location dataset: 6438
Institutions within 100 miles: 475


,Name of institution,Reported city,Reported state,Distance_Miles
3097,West Chester University of Pennsylvania,West Chester,PA,0.000000
2914,Cheyney University of Pennsylvania,Cheyney,PA,3.928814
2955,Immaculata University,Immaculata,PA,5.353610
4361,Chester County Intermediate Unit,Downingtown,PA,5.632786
2896,Automotive Training Center-Exton,Exton,PA,7.622513
...,...,...,...,...
2079,American Institute,Clifton,NJ,99.704458
1484,Montgomery Beauty School,Silver Spring,MD,99.739643
2186,Associated Beth Rivkah Schools,Brooklyn,NY,99.831943
2386,St. Joseph's University-New York,Brooklyn,NY,99.849208


In [26]:
df = df.rename(columns={'Unnamed: 0': 'University'})

# Clean university names in the original dataset
df['Match_Name'] = (
    df['University']
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace(r'[^a-z0-9 ]', '', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
)

# Clean institution names in the NCES dataset
locations['Match_Name'] = (
    locations['Name of institution']
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace(r'[^a-z0-9 ]', '', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
)

# Keep NCES institutions within 100 miles
nearby = locations[locations['Distance_Miles'] <= 100].copy()

# Match the two datasets and keep original college data
df_100mi = df.merge(
    nearby[['Match_Name', 'Distance_Miles']],
    on='Match_Name',
    how='inner'
)

# Display matching universities
print("Number of matching universities:", len(df_100mi))
display(df_100mi)


Number of matching universities: 37


,University,Private,Apps,Accept,Enroll,Top10perc,Top25perc,F.Undergrad,P.Undergrad,Outstate,...,Books,Personal,PhD,Terminal,S.F.Ratio,perc.alumni,Expend,Grad.Rate,Match_Name,Distance_Miles
0,Albright College,Yes,1038,839,227,30,63,973,306,15595,...,300,500,79,84,11.3,23,11644,80,albright college,32.639008
1,Bryn Mawr College,Yes,1465,810,313,71,95,1088,16,18165,...,500,1200,100,100,12.3,49,17449,89,bryn mawr college,15.923765
2,Bucknell University,Yes,6548,3813,862,49,85,3316,31,18550,...,800,1200,95,97,14.2,36,13675,93,bucknell university,96.659524
3,Cedar Crest College,Yes,776,607,198,25,58,791,764,14340,...,500,1000,58,83,11.7,39,10961,74,cedar crest college,43.928400
4,Chestnut Hill College,Yes,342,254,126,25,64,518,232,10335,...,700,850,71,71,8.3,29,7729,73,chestnut hill college,21.608868
5,Dickinson College,Yes,3014,2539,487,31,68,1889,62,18700,...,595,1250,87,94,11.2,39,13861,87,dickinson college,86.311931
6,Drew University,Yes,2153,1580,321,56,84,1192,87,18432,...,520,660,93,97,10.2,28,14907,83,drew university,83.292036
7,Elizabethtown College,Yes,2417,1843,426,36,70,1476,299,14190,...,500,750,65,68,12.8,25,9815,81,elizabethtown college,54.341665
8,Gettysburg College,Yes,3596,2466,575,42,78,1944,46,19964,...,500,500,94,95,12.1,32,14720,83,gettysburg college,87.239556
9,Goucher College,Yes,1151,813,248,40,64,850,80,15588,...,500,1200,78,90,9.2,34,16623,77,goucher college,64.944196


In [28]:

# Save the filtered dataset as a new CSV file
df_100mi.to_csv('WCUPA_Universities_Within_100_Miles.csv', index=False)

print("New dataset saved successfully!")
print("Number of universities:", len(df_100mi))
print("Number of columns:", len(df_100mi.columns))

# Preview the new dataset
display(df_100mi.head())

# Check its dimensions
print(df_100mi.shape)

# Check column names
print(df_100mi.columns.tolist())


New dataset saved successfully!
Number of universities: 37
Number of columns: 21


,University,Private,Apps,Accept,Enroll,Top10perc,Top25perc,F.Undergrad,P.Undergrad,Outstate,...,Books,Personal,PhD,Terminal,S.F.Ratio,perc.alumni,Expend,Grad.Rate,Match_Name,Distance_Miles
0,Albright College,Yes,1038,839,227,30,63,973,306,15595,...,300,500,79,84,11.3,23,11644,80,albright college,32.639008
1,Bryn Mawr College,Yes,1465,810,313,71,95,1088,16,18165,...,500,1200,100,100,12.3,49,17449,89,bryn mawr college,15.923765
2,Bucknell University,Yes,6548,3813,862,49,85,3316,31,18550,...,800,1200,95,97,14.2,36,13675,93,bucknell university,96.659524
3,Cedar Crest College,Yes,776,607,198,25,58,791,764,14340,...,500,1000,58,83,11.7,39,10961,74,cedar crest college,43.928400
4,Chestnut Hill College,Yes,342,254,126,25,64,518,232,10335,...,700,850,71,71,8.3,29,7729,73,chestnut hill college,21.608868


(37, 21)
['University', 'Private', 'Apps', 'Accept', 'Enroll', 'Top10perc', 'Top25perc', 'F.Undergrad', 'P.Undergrad', 'Outstate', 'Room.Board', 'Books', 'Personal', 'PhD', 'Terminal', 'S.F.Ratio', 'perc.alumni', 'Expend', 'Grad.Rate', 'Match_Name', 'Distance_Miles']


In [29]:
# Dataset dimensions
print("Dataset Shape:", df_100mi.shape)

# Column names
print("\nColumns:")
print(df_100mi.columns.tolist())

# Data types and non-null counts
print("\nDataset Information:")
df_100mi.info()

# Summary statistics for numeric variables
display(df_100mi.describe().T)

Dataset Shape: (37, 21)

Columns:
['University', 'Private', 'Apps', 'Accept', 'Enroll', 'Top10perc', 'Top25perc', 'F.Undergrad', 'P.Undergrad', 'Outstate', 'Room.Board', 'Books', 'Personal', 'PhD', 'Terminal', 'S.F.Ratio', 'perc.alumni', 'Expend', 'Grad.Rate', 'Match_Name', 'Distance_Miles']

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37 entries, 0 to 36
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   University      37 non-null     object 
 1   Private         37 non-null     object 
 2   Apps            37 non-null     int64  
 3   Accept          37 non-null     int64  
 4   Enroll          37 non-null     int64  
 5   Top10perc       37 non-null     int64  
 6   Top25perc       37 non-null     int64  
 7   F.Undergrad     37 non-null     int64  
 8   P.Undergrad     37 non-null     int64  
 9   Outstate        37 non-null     int64  
 10  Room.Board      37 non-null     int64

,count,mean,std,min,25%,50%,75%,max
Apps,37.0,4087.270270,3933.557195,323.000000,1456.000000,2417.00000,5220.000000,14446.000000
Accept,37.0,2353.918919,2080.460478,254.000000,1053.000000,1834.00000,2539.000000,10516.000000
Enroll,37.0,741.972973,700.542389,122.000000,321.000000,483.00000,865.000000,3252.000000
Top10perc,37.0,36.270270,19.970323,10.000000,24.000000,30.00000,42.000000,90.000000
Top25perc,37.0,65.756757,18.077619,31.000000,56.000000,63.00000,78.000000,100.000000
F.Undergrad,37.0,3071.189189,3168.060183,393.000000,1131.000000,1889.00000,3566.000000,14130.000000
P.Undergrad,37.0,896.108108,1197.471799,4.000000,80.000000,306.00000,1530.000000,4522.000000
Outstate,37.0,14327.864865,4209.468468,3648.000000,12000.000000,15276.00000,17748.000000,19964.000000
Room.Board,37.0,5348.621622,898.074340,3400.000000,4755.000000,5318.00000,5740.000000,7270.000000
Books,37.0,532.162162,124.010195,275.000000,450.000000,500.00000,600.000000,800.000000


### Key Findings 

Geographic filtering: The original university dataset was matched with NCES location data to identify institutions within 100 miles of WCUPA.

Dataset shape: After geographic filtering and matching, the final dataset contained 37 universities and 21 columns.

Matching quality: University names were standardized to improve matching between datasets. Unmatched institutions and duplicate matches were reviewed to help ensure the sample was accurate.

### 2.5 Data Quality Review

### 2.5.1 Standard Quality Checks

In [30]:
# -----------------------------
# Standard Data Quality Review
# -----------------------------

# Missing values by column
missing_summary = pd.DataFrame({
    'Missing Values': df_100mi.isnull().sum(),
    'Missing (%)': (
        df_100mi.isnull().sum() / len(df) * 100
    ).round(2)
})

missing_summary = missing_summary.sort_values(
    by='Missing (%)',
    ascending=False
)

print("Missing Values:")
display(missing_summary)

# Duplicate observations
print(f"\nDuplicate Rows: {df_100mi.duplicated().sum()}")

# Identify categorical variables
categorical_cols = df.select_dtypes(include='object').columns

# Review categorical values for consistency
print("\nCategorical Variables:")
for col in categorical_cols:
    print(f"{col}: {df_100mi[col].unique()}")

# Check for leading/trailing whitespace
whitespace_issues = {
    col: (df_100mi[col] != df_100mi[col].str.strip()).sum()
    for col in categorical_cols
}

print("\nLeading/Trailing Whitespace:")
display(
    pd.Series(
        whitespace_issues,
        name="Whitespace Issues"
    ).sort_values(ascending=False)
)

# Check for blank strings
blank_strings = {
    col: df_100mi[col].str.strip().eq('').sum()
    for col in categorical_cols
}

print("\nBlank String Values:")
display(
    pd.Series(
        blank_strings,
        name="Blank Strings"
    ).sort_values(ascending=False)
)

# Check for common non-standard missing-value representations
possible_missing_values = [
    '?',
    '-999',
    'NULL',
    'N/A',
    'n/a',
    'None',
    'none'
]

nonstandard_missing = {
    col: (
        df_100mi[col]
        .astype(str)
        .str.strip()
        .isin(possible_missing_values)
        .sum()
    )
    for col in categorical_cols
}

print("\nPotential Non-Standard Missing Values:")
display(
    pd.Series(
        nonstandard_missing,
        name="Potential Non-Standard Missing Values"
    ).sort_values(ascending=False)
)

# Identify variables with no variation
zero_variance_cols = [
    col for col in df.columns
    if df_100mi[col].nunique() <= 1
]

print("\nZero-Variance Variables:")
print(zero_variance_cols)

Missing Values:


,Missing Values,Missing (%)
University,0,0.0
Books,0,0.0
Match_Name,0,0.0
Grad.Rate,0,0.0
Expend,0,0.0
perc.alumni,0,0.0
S.F.Ratio,0,0.0
Terminal,0,0.0
PhD,0,0.0
Personal,0,0.0



Duplicate Rows: 0

Categorical Variables:
University: ['Albright College' 'Bryn Mawr College' 'Bucknell University'
 'Cedar Crest College' 'Chestnut Hill College' 'Dickinson College'
 'Drew University' 'Elizabethtown College' 'Gettysburg College'
 'Goucher College' 'Johns Hopkins University' "King's College"
 'La Salle University' 'Lafayette College' 'Lebanon Valley College'
 'Lehigh University' 'Lincoln University' 'Montclair State University'
 'Muhlenberg College' 'New Jersey Institute of Technology'
 'New York University' 'Pace University' 'Princeton University'
 'Rider University' 'Seton Hall University' "St. John's College"
 'Stevens Institute of Technology' 'Susquehanna University'
 'University of Delaware' 'University of Pennsylvania' 'Ursinus College'
 'Villanova University' 'Wagner College' 'Washington College'
 'Widener University' 'Wilkes University' 'York College of Pennsylvania']
Private: ['Yes' 'No']
Match_Name: ['albright college' 'bryn mawr college' 'bucknell universit

University    0
Private       0
Match_Name    0
Name: Whitespace Issues, dtype: int64


Blank String Values:


University    0
Private       0
Match_Name    0
Name: Blank Strings, dtype: int64


Potential Non-Standard Missing Values:


University    0
Private       0
Match_Name    0
Name: Potential Non-Standard Missing Values, dtype: int64


Zero-Variance Variables:
[]


### Key Findings

- No duplicate rows were identified in the dataset.
- Categorical variables, including University, Private, and Match_Name, contain consistent values with no detected whitespace issues, blank strings, or obvious non-standard missing-value representations.
- No zero-variance variables were identified, indicating that all variables have some variation in their values.
- The university names and their standardized matching names appear consistently formatted, supporting the process of matching university records across datasets.
- Overall, the dataset demonstrates good data quality based on the checks performed and requires minimal categorical data cleaning.

### Key Takeaway:
The dataset appears clean and suitable for continued analysis. No duplicate rows, categorical formatting issues, or zero-variance variables were detected. The next step is to verify that university names match correctly across datasets and proceed with data preparation for analyzing university graduation rates.

## Logical Validity Checks

Logical validity checks were performed to identify values that may be impossible, inconsistent, or suspicious within the university dataset. Because this study focuses on universities located within 100 miles of WCUPA, these checks are applied only to the selected regional sample.

The checks examine application and enrollment counts, percentages, tuition and expenditure values, graduation rates, duplicate university names, and missing values. These checks help identify potential data quality issues before analyzing graduation rates and their relationship with university characteristics.

Any suspicious observations will be investigated before deciding whether they should be corrected, retained, or excluded from the analysis.

In [38]:

# -----------------------------
# Logical Validity Checks
# -----------------------------

# Check whether values are logically valid
validity_checks = {
    'Negative Applications': (df_100mi['Apps'] < 0).sum(),

    'Accepted Applications Greater Than Total Applications': (
        df_100mi['Accept'] > df_100mi['Apps']
    ).sum(),

    'Enrolled Students Greater Than Accepted Students': (
        df_100mi['Enroll'] > df_100mi['Accept']
    ).sum(),

    'Top 10 Percent Greater Than Top 25 Percent': (
        df_100mi['Top10perc'] > df_100mi['Top25perc']
    ).sum(),

    'Graduation Rate Below 0 or Above 100': (
        (df_100mi['Grad.Rate'] < 0) | (df_100mi['Grad.Rate'] > 100)
    ).sum(),

    'PhD Percentage Below 0 or Above 100': (
        (df_100mi['PhD'] < 0) | (df_100mi['PhD'] > 100)
    ).sum(),

    'Terminal Degree Percentage Below 0 or Above 100': (
        (df_100mi['Terminal'] < 0) | (df_100mi['Terminal'] > 100)
    ).sum(),

    'PhD Percentage Greater Than Terminal Percentage': (
        df_100mi['PhD'] > df_100mi['Terminal']
    ).sum(),

    'Negative Tuition Values': (df_100mi['Outstate'] < 0).sum(),

    'Negative Expenditure Values': (df_100mi['Expend'] < 0).sum(),

    'Duplicate University Names': df_100mi['University'].duplicated().sum()
}

# Display the results in a table
validity_results = pd.DataFrame.from_dict(
    validity_checks,
    orient='index',
    columns=['Invalid Observations']
)

display(validity_results)


,Invalid Observations
Negative Applications,0
Accepted Applications Greater Than Total Applications,0
Enrolled Students Greater Than Accepted Students,0
Top 10 Percent Greater Than Top 25 Percent,0
Graduation Rate Below 0 or Above 100,0
PhD Percentage Below 0 or Above 100,0
Terminal Degree Percentage Below 0 or Above 100,0
PhD Percentage Greater Than Terminal Percentage,1
Negative Tuition Values,0
Negative Expenditure Values,0


In [37]:


# Identify universities where PhD percentage exceeds
# terminal degree percentage
print("PhD Percentage Greater Than Terminal Percentage:")
display(
    df_100mi.loc[
        df_100mi['PhD'] > df_100mi['Terminal'],
        ['University', 'PhD', 'Terminal']
    ]
)


PhD Percentage Greater Than Terminal Percentage:


,University,PhD,Terminal
34,Widener University,88,86


### Key Findings
- No negative application counts were identified, and accepted applications did not exceed total applications.
- No cases were found where enrolled students exceeded accepted students or where the top 10% student percentage exceeded the top 25% percentage.
- Graduation rates, PhD percentages, and terminal degree percentages were within the expected range of 0–100%.
- One potential logical inconsistency was identified where the PhD percentage exceeded the terminal degree percentage. This observation should be reviewed to determine whether it reflects a data quality issue or a difference in how the variables are defined.
- No negative tuition or expenditure values were detected.
- No duplicate university names were identified.
### Key Takeaway:
The dataset passed most logical validity checks, with no major inconsistencies identified across application counts, enrollment, graduation rates, tuition, or expenditures. One observation involving the PhD and terminal degree percentages requires further investigation and will be reviewed during the Data Preparation phase.

### 2.6 Target Variable Analysis 

The target variable analysis examines university graduation rates to establish a baseline before exploring potential factors associated with student outcomes.
This analysis focuses on:

- The overall distribution of university graduation rates across the dataset
- The average, median, minimum, and maximum graduation rates
- The universities with the highest and lowest reported graduation rates
- Whether the target variable contains unusual or invalid values that may affect the analysis

Understanding the distribution of graduation rates provides an important baseline for the analysis and directly addresses the business question:
What is the current level and distribution of graduation rates across universities, and how does WCU compare with its peer institutions?
The distribution of the target variable is also important for modeling. If graduation rates are concentrated within a particular range or contain unusual values, this may influence how the results are interpreted and which modeling techniques are appropriate. This analysis will help establish a foundation for examining potential relationships between graduation rates and university characteristics, such as instructional expenditure, tuition, and student-to-faculty ratio. 